In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Basic
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, KFold

# Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

# Metrics
from sklearn.metrics import mean_squared_error


In [ ]:
# Load datasets
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
sample_submission = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

# Display first few rows
display(train.head())
display(test.head())
display(sample_submission.head())


In [ ]:
sns.histplot(train["SalePrice"], kde=True)
plt.title("SalePrice Distribution")
plt.show()

# Check missing values
missing = train.isnull().sum().sort_values(ascending=False)
print("Top missing values:\n", missing[missing > 0].head(20))

# Scatterplot for outliers
sns.scatterplot(x=train["GrLivArea"], y=train["SalePrice"])
plt.title("GrLivArea vs SalePrice")
plt.show()

In [ ]:


from scipy.stats import skew   
# Drop obvious outliers
train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index)

# Log-transform target
train["SalePrice"] = np.log1p(train["SalePrice"])

# Save target
y = train["SalePrice"]

# Concatenate train+test for consistent processing
train_features = train.drop("SalePrice", axis=1)
all_data = pd.concat([train_features, test], axis=0)

# Handle missing values
for col in all_data.columns:
    if all_data[col].dtype == "object":
        all_data[col] = all_data[col].fillna("None")
    else:
        all_data[col] = all_data[col].fillna(all_data[col].median())

# Log-transform skewed numeric features
numeric_feats = all_data.dtypes[all_data.dtypes != "object"].index
skewness = all_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewed_feats = skewness[skewness > 0.75].index

for feat in skewed_feats:
    all_data[feat] = np.log1p(all_data[feat])

# One-hot encode categorical features
all_data = pd.get_dummies(all_data)

# Split back into train/test
X = all_data[:train.shape[0]]
X_test = all_data[train.shape[0]:]

print("Processed train shape:", X.shape)
print("Processed test shape:", X_test.shape)


In [ ]:
# ==============================
# Section 3: Model Selection & Training
# ==============================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

# Models
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Cross-validation setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)

def rmse_cv(model):
    """Cross-validated RMSE"""
    rmse = -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=kf)
    return rmse.mean()

# Candidate models
models = {
    "Linear Regression": LinearRegression(),
    "Neural Net": MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_state=42)
}

# Evaluate each
results = {}
for name, model in models.items():
    score = rmse_cv(model)
    results[name] = score
    print(f"{name}: {score:.4f}")

# Sort results
results = dict(sorted(results.items(), key=lambda x: x[1]))
print("\nModel ranking (best to worst):")
for name, score in results.items():
    print(f"{name}: {score:.4f}")


In [ ]:
importances = models["XGBoost"].fit(X, y).feature_importances_
feat_importances = pd.Series(importances, index=X.columns)
top_feats = feat_importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10,6))
sns.barplot(x=top_feats.values, y=top_feats.index, palette="viridis")
plt.title("Top 20 Important Features (XGBoost)")
plt.show()


In [ ]:
from sklearn.linear_model import Ridge

# Train one model for visualization
model = Ridge(alpha=10).fit(X, y)
preds = model.predict(X)
residuals = y - preds

plt.figure(figsize=(8,5))
sns.histplot(residuals, kde=True, bins=40)
plt.title("Residual Distribution")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(preds, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Predictions")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()


In [ ]:
# Train final model (replace with your best model)
final_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
final_model.fit(X, y)

# Predict test set
final_preds = final_model.predict(X_test)

# Inverse log1p transform
final_preds = np.expm1(final_preds)

# Create submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv created!")
